# 05 — Probability Calibration, Model Selection and Decision Analysis

1. **Calibration.** No calibration, sigmoid (Platt) and isotonic maps are fitted on 5-fold **out-of-fold training** scores (`CalibratedClassifierCV(ensemble=False)`). The method is chosen on validation.
2. **Model selection.** A rule declared before the final run (`src/credit_risk/selection.py`).
3. **Decision analysis.** F2-optimal thresholds with and without a capacity constraint, plus an *illustrative* cost analysis.

The test set is not used.

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
from credit_risk.config import load_config, resolve_path
BLUE, ORANGE, AQUA, YELLOW, GREY, INK, MUTED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#9a9a9a", "#1f1f1f", "#6b6b6b"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e6", "axes.edgecolor": MUTED, "axes.labelcolor": INK,
                     "axes.axisbelow": True, "figure.dpi": 110})
FIG_DIR = resolve_path("reports/figures")
M = resolve_path("artifacts/metrics")
pd.set_option("display.width", 200); pd.set_option("display.precision", 4); pd.set_option("display.max_rows", 60)
SKIP = {"summary", "comparisons", "selection"}
records = {p.stem: r for p in M.glob("*.json") if p.stem not in SKIP
           for r in [json.loads(p.read_text())] if r.get("evaluated_on") == "validation" and "model" in r}
summary = pd.DataFrame(json.loads((M / "summary.json").read_text())["models"]).set_index("model")
comparisons = pd.DataFrame(json.loads((M / "comparisons.json").read_text()))

def family(name):
    spec = records[name]["specification"]
    return spec.get("inherits_from", name).split("__")[0] if spec.get("inherits_from") else name

def strategy(name):
    rec = records[name]; spec = rec["specification"]; p = spec.get("params", {})
    if rec["calibration"] != "none": return f"calibrated ({rec['calibration']})"
    if spec.get("sampler"): return spec["sampler"]["name"]
    if p.get("class_weight") or p.get("scale_pos_weight"): return "class weights"
    if spec["estimator"] in {"balanced_random_forest", "easy_ensemble", "rusboost"}: return "imbalance-aware ensemble"
    return "none"


## 1. Calibration comparison

In [ ]:
cal_rows = []
for name, rec in records.items():
    base = name.split("__")[0]
    if not (("__" in name) or any(n.startswith(name + "__") for n in records)): continue
    pm = rec["probability_metrics"]
    cal_rows.append({"base_model": base, "calibration": rec["calibration"], "brier": pm["brier"], "log_loss": pm["log_loss"],
                     "ece": pm["expected_calibration_error"], "mean_pd": pm["mean_predicted_pd"], "roc_auc": pm["roc_auc"], "pr_auc": pm["pr_auc"]})
cal = pd.DataFrame(cal_rows).sort_values(["base_model", "calibration"]).set_index(["base_model", "calibration"])
cal

In [ ]:
def reliability(ax, names, title):
    ax.plot([0, 1], [0, 1], color=MUTED, ls="--", lw=1)
    for name, color in zip(names, [GREY, BLUE, ORANGE, AQUA]):
        t = pd.DataFrame(records[name]["calibration_table"]); ece = records[name]["probability_metrics"]["expected_calibration_error"]
        ax.plot(t["mean_predicted"], t["observed_rate"], color=color, lw=2, marker="o", ms=4, label=f"{name} (ECE {ece:.3f})")
    ax.set(title=title, xlabel="Mean predicted PD", ylabel="Observed default rate", xlim=(0, 1), ylim=(0, 1)); ax.legend(frameon=False, fontsize=7)
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
reliability(axes[0], ["xgboost", "xgboost__sigmoid", "xgboost__isotonic"], "XGBoost: calibration methods")
reliability(axes[1], ["xgboost_smotenc", "xgboost_smotenc__isotonic", "balanced_random_forest", "balanced_random_forest__isotonic"], "Resampled models before / after isotonic")
fig.tight_layout(); fig.savefig(FIG_DIR / "05_reliability.png", bbox_inches="tight")

### Before vs after calibration, same metrics, same validation set
Raw scores, isotonic recalibration (fitted on out-of-fold **training** scores) and the analytic prior correction (no fitted parameters), side by side.

In [ ]:
from credit_risk.calibration import prior_correction
from credit_risk.evaluation import probability_metrics
rows = []
for base in ["xgboost", "xgboost_weighted", "xgboost_smotenc", "balanced_random_forest", "logreg"]:
    for label, name in [("raw", base), ("isotonic", f"{base}__isotonic"), ("sigmoid", f"{base}__sigmoid")]:
        if name in records:
            pm = records[name]["probability_metrics"]
            rows.append({"model": base, "version": label, **{k: pm[k] for k in ["roc_auc", "pr_auc", "brier", "log_loss", "expected_calibration_error", "mean_predicted_pd"]}})
    if base != "xgboost" and base != "logreg":
        d = pd.read_csv(resolve_path(f"artifacts/predictions/validation_{base}.csv"))
        pm = probability_metrics(d["default_next_month"], prior_correction(d["pd"], records["dummy_prior"]["probability_metrics"]["mean_predicted_pd"], 0.5))
        rows.append({"model": base, "version": "prior correction (analytic)", **{k: pm[k] for k in ["roc_auc", "pr_auc", "brier", "log_loss", "expected_calibration_error", "mean_predicted_pd"]}})
pd.DataFrame(rows).set_index(["model", "version"])

## 2. Model selection (pre-declared rule)

In [ ]:
from credit_risk.selection import select_model
selection = select_model(M, resolve_path("artifacts/predictions"))
print(json.dumps(selection["rule"], indent=2))
print("Best PR-AUC among eligible models:", selection["best_pr_auc_model"])
print("Selected (parsimony):", selection["selected_model"])
print("Statistically equivalent:", selection["equivalent_models"])
pd.DataFrame(selection["ranking"]).head(15)

## 3. Decision analysis for the selected model

In [ ]:
from credit_risk.decision_analysis import expected_cost, threshold_table
chosen = selection["selected_model"]
rec = records[chosen]
pred = pd.read_csv(resolve_path(f"artifacts/predictions/validation_{chosen}.csv"))
tt = threshold_table(pred["default_next_month"], pred["pd"])
rules = {"F2 (unconstrained)": rec["threshold_selection"]["threshold"],
         "F2, at most 30% flagged": rec["threshold_selection_capacity"]["threshold"], "0.5 (reference)": 0.5}
pick = lambda t: tt.iloc[(tt["threshold"] - t).abs().argmin()]
pd.DataFrame({k: pick(v) for k, v in rules.items()}).T[["threshold", "tp", "fp", "tn", "fn", "precision", "recall", "f1", "f2", "flagged_rate"]]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for col, color, label in [("recall", BLUE, "Recall"), ("precision", ORANGE, "Precision"), ("f2", AQUA, "F2"), ("flagged_rate", GREY, "Share flagged")]:
    ax.plot(tt["threshold"], tt[col], color=color, lw=2, label=label)
ax.axhline(0.30, color=GREY, ls="--", lw=0.8); ax.text(0.86, 0.31, "30% capacity", fontsize=7, color=MUTED, ha="right")
for name, t in rules.items():
    ax.axvline(t, color=MUTED, lw=1, ls=":"); ax.text(t + 0.005, 1.02, name.split(",")[0].split(" (")[0] + (" cap" if "30%" in name else ""), fontsize=7, color=MUTED)
ax.set(title=f"Decision metrics vs threshold — {chosen} (validation)", xlabel="Threshold on predicted PD", ylabel="Value", xlim=(0, 0.9), ylim=(0, 1.08))
ax.legend(frameon=False, fontsize=8, ncol=4, loc="lower center", bbox_to_anchor=(0.5, -0.38))
fig.tight_layout(); fig.savefig(FIG_DIR / "05_threshold_curves.png", bbox_inches="tight")

### Illustrative cost analysis
**Hypothetical assumption, not bank data:** a missed defaulter (FN) costs 5 units and a false alarm (FP) costs 1 unit. The dataset has no exposure, LGD or margin information, so these figures show how the optimal threshold moves with the cost ratio. They are not estimated losses.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
out = {}
for ratio, color in [(2, BLUE), (5, ORANGE), (10, AQUA)]:
    c = expected_cost(tt, cost_false_negative=ratio, cost_false_positive=1.0)
    best = c.loc[c["illustrative_cost_per_client"].idxmin()]
    out[f"FN:FP = {ratio}:1"] = best[["threshold", "recall", "precision", "flagged_rate", "illustrative_cost_per_client"]]
    ax.plot(c["threshold"], c["illustrative_cost_per_client"], color=color, lw=2, label=f"FN:FP cost = {ratio}:1 (min at {best['threshold']:.2f})")
    ax.scatter([best["threshold"]], [best["illustrative_cost_per_client"]], color=color, s=30, zorder=3)
ax.set(title="Illustrative cost per client vs threshold (hypothetical costs)", xlabel="Threshold", ylabel="Cost per client (units)", xlim=(0, 0.9))
ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "05_illustrative_cost.png", bbox_inches="tight")
pd.DataFrame(out).T

For a calibrated PD, the cost-minimising threshold under the FN:FP ratio *r* is approximately 1 / (1 + r) (0.33, 0.17 and 0.09 for *r* = 2, 5, 10). The empirical minima (0.35, 0.15, 0.12) land close to these values, which is another check that the selected model's PDs are well calibrated.

## Findings

**Calibration**
- **XGBoost and Logistic Regression are already well calibrated without post-processing** (validation ECE 0.015 and 0.017). Neither sigmoid nor isotonic calibration improves their Brier score significantly.
- Isotonic calibration lowers PR-AUC significantly (XGBoost −0.012, LR −0.011). Its step function creates ties among the top-ranked clients.
- For **resampled or reweighted models, isotonic calibration is essential**. It brings ECE from about 0.18–0.20 down to about 0.01. Even after recalibration, these models do not beat the plain XGBoost on PR-AUC.

**Selection (pre-declared rule)**
- The best eligible PR-AUC comes from `xgboost__sigmoid`, which has the same ranking as `xgboost` because a monotone map doesn't change the order.
- Under the parsimony step, `xgboost`, `xgboost__sigmoid` and `xgboost_tomek_links` are statistically equivalent. The rule selects **`xgboost` without calibration or resampling**.

**Decision rules for the selected model (validation)**

| Rule | Threshold | Recall | Precision | Share flagged |
|---|---|---|---|---|
| F2, unconstrained | 0.135 | 0.849 | 0.333 | 56% |
| F2, ≤ 30% flagged | 0.230 | 0.622 | 0.470 | 29% |
| 0.5 (reference only) | 0.500 | 0.360 | 0.689 | 12% |

Recall can only rise by flagging more clients. The 30% capacity rule catches about 62% of defaulters with nearly one alert in two correct. That compares with 22% precision for random flagging.

**What the thresholds mean, and what they don't.**
- Both thresholds are chosen on validation and then **frozen** before the test evaluation (notebook 06, `frozen_decision.json`).
- The unconstrained F2 rule reflects a stated preference for recall. The 30% capacity rule reflects an assumed review capacity, which is an illustrative operating assumption, not a business figure.
- A PD threshold is **a screening rule, not a credit decision.** An actual approval or limit policy would also need exposure (EAD), loss given default (LGD), pricing and margin, error costs, regulatory and business rules, and fairness review. None of these are available in this dataset.